# Lesson 5.2 · Scale, distance and the curse of dimensionality

All of KNN depends on **one** thing: what you mean by "near". The distance sums the contributions of all the features, and so the features with a large scale or no information dominate
the result. This lesson shows two very concrete effects.

**What you will learn**

- Why scaling is mandatory and not a detail, and what different metrics do.
- Why adding useless features **ruins** KNN, and what it means that in many dimensions "everyone is far".

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine, make_moons
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(X.shape, "| std dev of the features: min", X.std().min().round(2), "max", X.std().max().round(1))
X.std().round(2).sort_values(ascending=False).head(4)

## The scale decides who counts

The Euclidean distance sums the squares of the differences. In the wine dataset `proline` has a standard deviation of about 315, the second (`magnesium`) 14 and all the others about 3 or less: in the distance
`proline` does almost **everything** and the other 12 columns weigh very little.

In [ ]:
V = X.to_numpy()
squares = (V[:, None, :] - V[None, :, :]) ** 2                   # contribution of every feature to the squared distance, for every pair of wines
print("mean share of proline in the squared distance:", round(squares[:, :, list(X.columns).index("proline")].sum() / squares.sum(), 4))

without = KNeighborsClassifier(7)
with_scaler = make_pipeline(StandardScaler(), KNeighborsClassifier(7))
print("accuracy without scaling:", cross_val_score(without, X, y, cv=cv).mean().round(3), "| with StandardScaler:", cross_val_score(with_scaler, X, y, cv=cv).mean().round(3))

With scaling the accuracy goes up by almost thirty points (from 0.66 to 0.95). It is not a novelty of KNN: the same holds for all distance-based models (and for PCA, module 6). The scaler goes inside the `Pipeline`, so it is learned
on the training part of every fold alone.

## Metrics

The default metric is Minkowski with `p=2` (Euclidean). Other useful choices: `manhattan` (sum of absolute values: less sensitive to large gaps on a single feature), `chebyshev` (the maximum gap), `cosine`
(angle between the vectors: ignores the length, useful for text and counts). Scaling changes much more than the metric, as you will see in exercise F1.

## More features does not mean more information

Let's add columns of pure noise to a 2-dimensional problem (the moons) and look at the accuracy.

In [ ]:
rng = np.random.default_rng(0)
Xm, ym = make_moons(1500, noise=0.25, random_state=0)
rows = []
for n_noise in [0, 2, 5, 10, 20, 50, 100]:
    Xr = np.c_[Xm, rng.normal(size=(len(Xm), n_noise))]
    rows.append({"noise columns": n_noise, "accuracy (k=5)": cross_val_score(KNeighborsClassifier(5), Xr, ym, cv=cv).mean()})
pd.DataFrame(rows).round(3).set_index("noise columns").T

The two useful dimensions do not change, but every noise column adds a random term to the squared distance: the "true" neighbors are covered by the accumulated random distance. With 100 useless columns the accuracy falls from 0.93 to 0.57, little better than chance (0.5).
It is the **curse of dimensionality**: in many dimensions the distances between points concentrate (the nearest neighbor is almost as far as the farthest) and "near" loses meaning.

> **Common mistake.** Adding all the available features "because maybe they help". With distance-based models every useless feature has a cost; feature selection (or a PCA, module 6) is part of the model.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Do I need to scale the dummy (0/1) variables of a one-hot before the KNN?</summary>

It depends on how much you want them to weigh: a 0/1 dummy has a smaller scale than a standardized feature, and a category with many levels adds many dummies to the distance. A common choice is to scale the numerics, leave the dummies at 0/1 and check with cross-validation whether a different weight helps.

</details>

<details><summary>Why does the cosine work with text?</summary>

Because a long document and a short one on the same topic have count vectors with the same direction but different lengths: the Euclidean distance considers them far, the cosine (which looks only at the angle) near.

</details>